## STACKING MODELS WITH MLXtend

In [1]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.linear_model import RidgeCV, LinearRegression
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, StackingRegressor

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error as MSE, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures, FunctionTransformer 
from sklearn.compose import ColumnTransformer

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectFromModel, SelectKBest, f_regression, VarianceThreshold
from sklearn.impute import SimpleImputer


In [2]:
data = pd.read_csv("cleaned_df.csv")
data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
data.shape

(1460, 81)

In [4]:
#remove outliers
def remove_outliers_iqr(df, cols, multiplier=1.5):
    """Remove rows that have outliers in specified columns using IQR rule."""
    df_clean = df.copy()
    for col in cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.85)
        IQR = Q3 - Q1
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        mask = (df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)
        df_clean = df_clean[mask]
    return df_clean

# Apply only to numerical features
numeric_cols = data[["GrLivArea", "LotFrontage", "LotArea", "MasVnrArea", "SalePrice"]]
df = remove_outliers_iqr(data, numeric_cols, multiplier=1.5)

print(f"Original shape: {data.shape}, after outlier removal: {df.shape}")

df.describe()

Original shape: (1460, 81), after outlier removal: (1363, 81)


,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,MiscVal,MoSold,YrSold,SalePrice
count,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,...,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000,1363.000000
mean,731.300807,57.105649,56.337491,9274.330154,6.016141,5.582539,1970.733676,1984.396919,84.093177,3.367572,...,44.727806,21.986060,3.423331,14.230374,1.820249,0.008804,45.703595,6.343360,2007.821717,172052.598679
std,422.400975,42.517223,31.697644,3316.586359,1.312329,1.111766,30.144784,20.722278,134.537215,0.545201,...,64.447830,59.667217,29.744318,54.168184,33.856059,0.171146,512.895137,2.693238,1.331224,64083.723208
min,1.000000,20.000000,0.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,2.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,364.500000,20.000000,42.500000,7398.000000,5.000000,5.000000,1953.000000,1966.000000,0.000000,3.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,128500.000000
50%,734.000000,50.000000,62.000000,9230.000000,6.000000,5.000000,1972.000000,1993.000000,0.000000,3.000000,...,24.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,158900.000000
75%,1094.500000,70.000000,78.000000,11199.000000,7.000000,6.000000,2000.000000,2003.500000,145.000000,4.000000,...,65.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,204375.000000
max,1460.000000,190.000000,149.000000,21000.000000,10.000000,9.000000,2010.000000,2010.000000,640.000000,5.000000,...,547.000000,386.000000,508.000000,480.000000,738.000000,4.000000,15500.000000,12.000000,2010.000000,402861.000000


In [5]:
null_values = df.isnull().sum().sort_values(ascending = False)
print(null_values[null_values > 0])

MiscFeature     1311
Alley           1277
Fence           1088
MasVnrType       831
GarageFinish      79
GarageType        79
Electrical         1
dtype: int64


In [6]:
df.dropna(axis = 'columns')
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

## Feature Engineering and Preprocessing Pipeline

#### Automatic feature creator

Define a class to create features automatically whenever there is a new X input.

In [7]:
# class for feature creation
class FeatureCreator(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None): # temporarily create features to be used in column transformation
        if y is not None:
            df = X.copy()
            df['SalePrice'] = y
            df['price_sqft'] = df['SalePrice'] / df['GrLivArea'].replace(0, np.nan)

            self.neighborhood_avg_price_sqft = df.groupby('Neighborhood')['price_sqft'].mean().to_dict()
            self.nbhood_avghouse_price = df.groupby("Neighborhood")["SalePrice"].mean().to_dict()
            self.SubClass_avg_price_sqft = df.groupby("MSSubClass")["price_sqft"].mean().to_dict()
            self.Zoning_avg_price_sqft = df.groupby("MSZoning")["price_sqft"].mean().to_dict()
            self.yearbuilt_avg_price = df.groupby("YearBuilt")["SalePrice"].mean().to_dict()
            df['age'] = df['YrSold'] - df['YearBuilt']
            df['age_afterRemodel'] = df['YrSold'] - df['YearRemodAdd']
            self.age_avg_price = df.groupby("age")["SalePrice"].mean().to_dict()
            self.age_afterRemodel_price_sqft = df.groupby("age_afterRemodel")["SalePrice"].mean().to_dict()
            self.rooms_avg_price = df.groupby("TotRmsAbvGrd")["SalePrice"].mean().to_dict()
            self.overalcond_avg_price = df.groupby("OverallCond")["SalePrice"].mean().to_dict()
        return self

    def transform(self, X):
        df = X.copy()

        # Basic features
        df['total_area'] = df.get('GrLivArea', 0) + df.get('TotalBsmtSF', 0) + df.get('GarageArea', 0)
        df['total_bathrooms'] = (
            df.get('FullBath', 0) +
            0.5 * df.get('HalfBath', 0) +
            df.get('BsmtFullBath', 0) +
            0.5 * df.get('BsmtHalfBath', 0)
        )

        # Age features
        df['age'] = df.get('YrSold', 0) - df.get('YearBuilt', 0)
        df['age_afterRemodel'] = df.get('YrSold', 0) - df.get('YearRemodAdd', 0)

        # Mapping price new features — without overwriting originals
        df['nbd_avg_price_sqf'] = df['Neighborhood'].map(self.neighborhood_avg_price_sqft)
        df['nbd_avg_price_sqf'] = df['nbd_avg_price_sqf'].fillna(np.mean(list(self.neighborhood_avg_price_sqft.values())))

        df['nbd_avg_house_price'] = df['Neighborhood'].map(self.nbhood_avghouse_price)
        df['SubClass_avg_price_sqf'] = df['MSSubClass'].map(self.SubClass_avg_price_sqft)
        df['Zoning_avg_price_sqf'] = df['MSZoning'].map(self.Zoning_avg_price_sqft)
        df['yearbuilt_avg_price'] = df['YearBuilt'].map(self.yearbuilt_avg_price)
        df['age_avg_price'] = df['age'].map(self.age_avg_price)
        df['age_afterRemodel_price'] = df['age_afterRemodel'].map(self.age_afterRemodel_price_sqft)
        df['rooms_avg_price'] = df['TotRmsAbvGrd'].map(self.rooms_avg_price)
        df['overalcond_avg_price'] = df['OverallCond'].map(self.overalcond_avg_price)

        return df


In [8]:
# Second transformer: price-related features (requires target during fit)
#price related feature transformer
class TargetRelatedFeatureCreator(BaseEstimator, TransformerMixin):
    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y, index=X.index)

        # Compute price per square foot (per-row) once and store temporarily
        X["price_sqft"] = y / X["total_area"]

        # Store global average from TRAIN data only (scalar)
        self.avg_price_sqft_ = X["price_sqft"].mean()
        #price ratio
        self.price_sqft_ratio_ = X["price_sqft"] / self.avg_price_sqft_
        #Z-score normalization
        self.price_sqft_zscore_ = (X["price_sqft"] - self.avg_price_sqft_) / X["price_sqft"].std()
        #global average house price
        self.global_avg_house_price_ = y.mean()
       
        return self

    def transform(self, X):
        X = X.copy()

        # Global avg (scalar, same for all rows)
        X["avg_price_sqft"] = self.avg_price_sqft_

        #price ratio
        X["price_sqft_ratio"] = self.price_sqft_ratio_ 
        
        #Z-score normalization
        X["price_sqft_zscore"] = self.price_sqft_zscore_
       
        #global average house price
        X["avg_house_price"] = self.global_avg_house_price_
       
        return X

### Split the dataset

In [9]:
df = df.dropna(subset=['SalePrice'])

# Split features and target
X = df.drop(columns = ['SalePrice'])
X = X.dropna(axis = 'columns')

y = df['SalePrice']  # KEEP AS IS (raw)


# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature creation from training data for the sole purpose of selecting numerical features for columtransfomer
feature_creation = FeatureCreator()

X_train_created = feature_creation.fit_transform(X_train, y_train)

#  Use only the transformed X_train to define feature types
num_features = X_train_created.select_dtypes(include='number').columns.tolist()
cat_features = X_train_created.select_dtypes(include=['object', 'category']).columns.tolist()


#### Select polynomial features to be added

In [10]:
#Apply PolynomialFeatures only to a few important columns 

Selected_poly_features= ['LotFrontage', 'yearbuilt_avg_price', 'OverallQual', 'GrLivArea', 
                         'total_area', 'nbd_avg_price_sqf', 'rooms_avg_price', 'OverallCond', 
                         'age_avg_price', 'ExterQual', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 
                         '2ndFlrSF', 'BsmtFinSF1', 'nbd_avg_house_price', 'BsmtFinType2', 'FullBath', 
                         'HeatingQC', 'CentralAir', '1stFlrSF', 'BsmtFullBath', 'KitchenQual', 
                         'Fireplaces', 'age_afterRemodel_price', 'SubClass_avg_price_sqf', 'Zoning_avg_price_sqf', 
                         'overalcond_avg_price', 'total_bathrooms', 'BedroomAbvGr', 'TotRmsAbvGrd', 
                        'GarageArea', "LotArea"]


#### Full preprocessing pipeline

In [11]:
#polynomial features
poly_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
    ('variance_filter', VarianceThreshold(threshold=0.005)),  # Removes low-variance columns
    ('select', SelectKBest(score_func = f_regression, k = 13))
])

# Preprocessing for numeric features
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  MinMaxScaler())
])

# Preprocessing for categorical features
cat_processor = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown ='ignore', sparse_output=False))
])

#column transformer
preprocessor = ColumnTransformer(
    transformers = [
        ('poly', poly_pipeline, Selected_poly_features),
        ('num', num_pipeline, num_features),
        ('cat', cat_processor, cat_features)
    ])

# feature selector using a model (e.g., Ridge with L2 regularization)
#feature_selector = SelectFromModel(RidgeCV(alphas=np.logspace(-3, 3, 7)))

## Define the Models

In [32]:
#### Base regressors
xgb = XGBRegressor(n_estimators = 500, learning_rate = 0.03, max_depth = 2, 
                   random_state = 42, verbosity = 0, n_jobs = 4)

lgbm = LGBMRegressor(n_estimators = 500, learning_rate = 0.046, max_depth = 2, 
                     random_state = 42, verbose = 0, n_jobs = 4)

rf = RandomForestRegressor(n_estimators = 500, random_state = 42, n_jobs = 4)


ridge = RidgeCV(alphas = np.logspace(-3, 3, 7))
catboost = CatBoostRegressor(verbose = 0, learning_rate = 0.03, depth = 10, random_state = 42)

base_models = [('xgb', xgb), 
               ('rf', rf),
               ('lgbm', lgbm),
               ('catboost', catboost)
              ]

#meta_model = CatBoostRegressor(verbose = 0, learning_rate = 0.038, depth = 10, random_state = 42)
#meta_model = LinearRegression()
meta_model = RidgeCV(alphas = np.logspace(-3, 3, 7))

In [33]:
# stacking 
stacked_models = StackingRegressor(
    estimators = base_models,
    final_estimator = meta_model,
    cv = 5,  # Enables OOF prediction
    passthrough = False,  # Optional: If True, original features are passed to meta-model
    n_jobs = -1
)

## End-to-End ML Full Pipeline

In [34]:
# full pipeline
full_pipeline = Pipeline(steps=[
    ('feature_creator', feature_creation),  # custom feature engineer
    ('preprocessor', preprocessor),
    ('variance_threshold', VarianceThreshold(threshold=0.005)), 
    ('model', stacked_models)
])

## Fit and Predict

In [35]:
full_pipeline.fit(X_train, y_train)

y_pred = full_pipeline.predict(X_test)


## Model Evaluation

In [36]:
# Metrics
rmse = np.sqrt(MSE(y_test, y_pred))
rmse_log = np.sqrt(MSE(np.log(y_test), np.log(y_pred)))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Log RMSE on Test Set: {rmse_log:.4f}")
print(f"R² on Test Set: {r2:.4f}")
print(f"MAE on Test Set: {mae:.2f}")


Log RMSE on Test Set: 0.1193
R² on Test Set: 0.9136
MAE on Test Set: 11661.25


Log RMSE on Test Set: 0.1180

R² on Test Set: 0.9119

MAE on Test Set: 11673.90

## Combined Hyperperameter  for Base Estimators

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint, loguniform

# Assume `feature_creation`, `preprocessor`, `stacked_models`, `X_train`, `y_train`
# and the lists `Selected_poly_features`, `num_features`, `cat_features` are defined.

# Reconstructing the full pipeline for clarity.
# You do not need to re-run this part if already defined.
full_pipeline = Pipeline(steps=[
    ('feature_creator', feature_creation),
    ('preprocessor', preprocessor),
    ('variance_threshold', VarianceThreshold(threshold=0.005)),
    ('model', stacked_models)
])

# It's recommended to log-transform the target variable before tuning
y_train_log = np.log1p(y_train)

# Define the parameter distributions for RandomizedSearchCV
# The format is 'step_name__parameter_name' or 'step_name__sub_step__parameter_name'
param_distributions = {
    # Stacking Regressor parameters, accessed via the 'model' step
    'model__passthrough': [True, False],
    'model__final_estimator__alphas': loguniform(1e-3, 1e3), # For RidgeCV
    
    # Base model parameters within the stacking regressor
    'model__xgb__n_estimators': randint(100, 500),
    'model__xgb__learning_rate': uniform(0.01, 0.2),
    'model__xgb__max_depth': randint(3, 10),
    
    'model__lgbm__n_estimators': randint(100, 500),
    'model__lgbm__learning_rate': uniform(0.01, 0.2),
    'model__lgbm__num_leaves': randint(20, 50),
    
    'model__rf__n_estimators': randint(100, 500),
    'model__rf__max_features': ['sqrt', 'log2', None],
    
    'model__catboost__n_estimators': randint(100, 500),
    'model__catboost__learning_rate': uniform(0.01, 0.2),
    
    # Feature Selection within the preprocessor's 'poly' pipeline
    'preprocessor__poly__select__k': randint(5, 20),
    
    # Variance Threshold for the full pipeline
    'variance_threshold__threshold': uniform(0.001, 0.01)
}

# Create the RandomizedSearchCV object
random_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=50,  # Number of parameter settings that are sampled
    cv=5,
    scoring='neg_root_mean_squared_error',
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Fit the search to data
random_search.fit(X_train, y_train_log)

# Print the best parameters and best score
print("Best Parameters: ", random_search.best_params_)
print("Best Score (Log RMSE): ", -random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters:  {'model__catboost__learning_rate': 0.12237333842875472, 'model__catboost__n_estimators': 367, 'model__final_estimator__alphas': 0.37494206267589464, 'model__lgbm__learning_rate': 0.1601742135582995, 'model__lgbm__n_estimators': 316, 'model__lgbm__num_leaves': 22, 'model__passthrough': True, 'model__rf__max_features': 'log2', 'model__rf__n_estimators': 372, 'model__xgb__learning_rate': 0.11686549470611268, 'model__xgb__max_depth': 6, 'model__xgb__n_estimators': 467, 'preprocessor__poly__select__k': 13, 'variance_threshold__threshold': 0.004200496010306118}
Best Score (Log RMSE):  0.1094958312068998


### GridSearchCV

Narrow the search window with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    # Stacking Regressor parameters, accessed via the 'model' step
    'model__passthrough': [True],
    'model__final_estimator__alphas': [0.37], # Fine-tuning around the best value
    
    # Base model parameters
    #'model__xgb__n_estimators': [450, 460, 470],
    'model__xgb__learning_rate': [0.11, 0.117, 0.12],
    #'model__xgb__max_depth': [6],

    #'model__lgbm__n_estimators': [300, 310, 320],
    'model__lgbm__learning_rate': [0.15, 0.16, 0.17],
    #'model__lgbm__num_leaves': [21, 22, 23],
    
    #'model__rf__n_estimators': [360, 370, 380],
    'model__rf__max_features': ['log2'],
    
    #'model__catboost__n_estimators': [360, 370, 380],
    'model__catboost__learning_rate': [0.12, 0.122, 0.13],
    
    # Feature Selection within the preprocessor's 'poly' pipeline
    #'preprocessor__poly__select__k': [12, 13, 14],
    
    # Variance Threshold for the full pipeline
    'variance_threshold__threshold': [0.004, 0.005]
}

# Create the GridSearchCV object
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    verbose=2,
    n_jobs=-1
)

# Fit the search to data
grid_search.fit(X_train, y_train_log)

# Print the best parameters and best score
print("Best Parameters: ", grid_search.best_params_)
print("Best Score (Log RMSE): ", -grid_search.best_score_)

Fitting 5 folds for each of 4374 candidates, totalling 21870 fits


KeyboardInterrupt: 